# 01. Data Understanding & Exploratory Data Analysis (EDA)

This notebook **only explores** the Cupid Matchmaking Dataset (100,000 profile pairs): structure, distributions, the target `compatible`, correlations and demographic patterns.

It is **read-only**: the raw data is never modified, cleaned, encoded, split or saved here. Those tasks belong to the other notebooks (`02_preprocessing`, `03_logistic_regression`, ...).

## Step 0: Setup and Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RAW_PATH  = '../data/raw/cupid_algorithm_dataset.csv'
TARGET    = 'compatible'
LONGEVITY = 'relationship_longevity_months'


def text_columns(frame):
    """Columns that are neither numeric nor boolean (text / categorical)."""
    return [c for c in frame.columns
            if not pd.api.types.is_numeric_dtype(frame[c]) and not pd.api.types.is_bool_dtype(frame[c])]

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Dataset Shape: {df.shape}')
df.head()

## 1. Structure of the dataset

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Column groups (used by the plots below; df itself is not modified)
categorical  = text_columns(df)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
flag_cols    = [c for c in numeric_cols if set(df[c].dropna().unique()) <= {0, 1}]
continuous   = [c for c in numeric_cols if c not in flag_cols]
age_cols     = [c for c in ['a_age', 'b_age'] if c in df.columns]

print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print(f'Continuous/numeric: {len(continuous)} | 0/1 flags: {len(flag_cols)} | Categorical (text): {len(categorical)}')
print('Categorical columns:', categorical)

# Working copy for exploration-only columns (age categories, age gap)
eda = df.copy()

## 2. Target variable distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET, hue=TARGET, palette='Set2', legend=False)
plt.title('Target Variable Distribution (compatible)')
plt.xlabel('Compatible (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

print(df[TARGET].value_counts(normalize=True))

## 3. Distribution of the features (independent exploration)

In [ ]:
cols_to_plot = continuous[:12]
n = len(cols_to_plot)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
    for ax, col in zip(np.ravel(axes), cols_to_plot):
        sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax)
        ax.set_title(col)
    for ax in np.ravel(axes)[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical features: share of each category
for col in categorical:
    if df[col].nunique() <= 15:
        print(f'\n{col}:')
        print((df[col].value_counts(normalize=True, dropna=False) * 100).round(1).astype(str) + ' %')
    else:
        print(f'\n{col}: {df[col].nunique()} unique values')

## 4. Age analysis

In [ ]:
bins   = [0, 2, 12, 19, 35, 60, 150]
labels = ['Newborn', 'Child', 'Teenager', 'Young Adult', 'Senior Adult', 'Senior']
age_cat_cols = []
for col in age_cols:
    new = f'{col}_category'
    eda[new] = pd.cut(eda[col], bins=bins, labels=labels, right=False)
    age_cat_cols.append(new)

if {'a_age', 'b_age'} <= set(eda.columns):
    eda['age_gap'] = (eda['a_age'] - eda['b_age']).abs()

if age_cat_cols:
    fig, axes = plt.subplots(1, len(age_cat_cols), figsize=(6 * len(age_cat_cols), 4), squeeze=False)
    for ax, col in zip(axes[0], age_cat_cols):
        eda[col].value_counts().reindex(labels).plot(kind='bar', ax=ax, title=f'{col} distribution')
    plt.tight_layout()
    plt.show()
else:
    print('No age column found.')

## 5. Relationship longevity vs compatibility

In [ ]:
if LONGEVITY in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=TARGET, y=LONGEVITY, hue=TARGET, palette='Set2', legend=False)
    plt.title('Relationship Longevity (Months) by Compatibility')
    plt.show()
else:
    print(f'Column {LONGEVITY} not found.')

## 6. Feature correlations

In [ ]:
corr_cols = (numeric_cols + [TARGET])[:25]
corr = df[corr_cols].corr()
plt.figure(figsize=(max(8, len(corr_cols) * 0.55), max(6, len(corr_cols) * 0.45)))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=len(corr_cols) <= 14, fmt='.2f')
plt.title('Correlation matrix of numeric features')
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
high = pairs[pairs.abs() > 0.9]
print('Highly correlated pairs (|r| > 0.9):', high.to_dict() if len(high) else 'none')

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corrs = numeric_df.corr()[TARGET].sort_values(ascending=False)
print('Top Correlations with Target (compatible):')
print(corrs)

top = corrs.drop(TARGET).abs().sort_values(ascending=False).head(15).index
plt.figure(figsize=(8, 5))
corrs.drop(TARGET).loc[top].sort_values().plot(kind='barh')
plt.title('Top 15 correlations with the target')
plt.show()

## 7. Compatibility by group

As a reference, the overall compatibility rate is compared with each group; only differences larger than 5 points are printed.

In [ ]:
base_rate = eda[TARGET].mean()
print(f'Overall compatibility rate: {base_rate:.1%}\n')

group_cols = [c for c in categorical + age_cat_cols if eda[c].nunique() <= 15]
found = False
for col in group_cols:
    rate = eda.groupby(col, observed=True)[TARGET].mean()
    if rate.max() - rate.min() > 0.05:
        found = True
        print(f'{col}:')
        print((rate * 100).round(1).astype(str) + ' %\n')
if not found:
    print('No group shows a difference larger than 5 points.')

for col in age_cat_cols:
    rate = eda.groupby(col, observed=True)[TARGET].mean().reindex(labels).dropna()
    rate.plot(kind='bar', title=f'Compatibility rate by {col}', figsize=(7, 4))
    plt.ylabel('Compatibility rate')
    plt.show()

In [ ]:
# Mean of each continuous feature for compatible vs. not compatible pairs
means = df.groupby(TARGET)[continuous].mean().T
std = df[continuous].std().replace(0, np.nan)
means['std_diff'] = (means.get(1, np.nan) - means.get(0, np.nan)) / std
display(means.sort_values('std_diff', key=lambda s: s.abs(), ascending=False).head(15))

## 8. Interaction effects

In [ ]:
# Age gap vs compatibility
if 'age_gap' in eda.columns:
    gap_bins = pd.cut(eda['age_gap'], bins=[-0.1, 2, 5, 10, 20, 200],
                      labels=['0-2', '3-5', '6-10', '11-20', '20+'])
    gap_rate = eda.groupby(gap_bins, observed=True)[TARGET].mean()
    gap_rate.plot(kind='bar', title='Compatibility rate by age gap (years)', figsize=(7, 4))
    plt.ylabel('Compatibility rate')
    plt.show()
    print((gap_rate * 100).round(1).astype(str) + ' %')
else:
    print('No age gap available (columns a_age / b_age not found).')

In [ ]:
# Both partners older than 35 vs both younger
if {'a_age', 'b_age'} <= set(eda.columns):
    both_old   = eda[(eda['a_age'] > 35) & (eda['b_age'] > 35)][TARGET].mean()
    both_young = eda[(eda['a_age'] <= 35) & (eda['b_age'] <= 35)][TARGET].mean()
    mixed      = eda[((eda['a_age'] > 35) ^ (eda['b_age'] > 35))][TARGET].mean()
    print(f'Compatibility, both > 35 : {both_old:.1%}')
    print(f'Compatibility, both <= 35: {both_young:.1%}')
    print(f'Compatibility, mixed     : {mixed:.1%}')

In [ ]:
# Profile comparison: standardized means of the top features
top_feats = means['std_diff'].abs().sort_values(ascending=False).head(10).index.tolist()
if top_feats:
    z = (df[continuous] - df[continuous].mean()) / df[continuous].std().replace(0, np.nan)
    comp = z[top_feats].groupby(df[TARGET]).mean().T
    comp.columns = ['Not compatible' if c == 0 else 'Compatible' for c in comp.columns]
    comp.plot(kind='bar', figsize=(11, 5))
    plt.title('Standardized feature means by compatibility')
    plt.ylabel('Mean (z-score)')
    plt.tight_layout()
    plt.show()

## Summary

This notebook gave an overview of the dataset: its structure, the balance of the target `compatible`, the distribution of the features, their correlations with the target and a few group and interaction effects. Everything that modifies or models the data (quality treatment, encoding, train/test split, scaling, models) is done in the following notebooks.